In [69]:
import pandas as pd

In [70]:
df_sold = pd.read_csv("CRMLSSold_2.csv", low_memory = False)
df_listings = pd.read_csv("CRMLSListing_2.csv", low_memory = False)

In [71]:
df_sold.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName
0,ContraCosta,ContraCosta,"Tile,Wood",NaN,NaN,NaN,False,5000000.0,1095075487,2024-01-18,...,NaN,False,3.0,NaN,94595,NaN,30240.0,NaN,CRMLS,CRMLS_CCBE
1,NaN,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1079166779,2024-01-30,...,NaN,False,2.0,Palm Springs Unified,92262,NaN,13504.0,NaN,CRMLS,CRMLS_MLSL
2,Glendale,Southland,NaN,False,NaN,NaN,False,1890500.0,1075037759,2024-01-29,...,5.0,False,2.0,Los Angeles Unified,91356,0.0,17873.0,NaN,CRMLS,CRMLS_CRM
3,NorthSanLuisObispo,NorthSanLuisObispo,NaN,True,NaN,NaN,False,2100000.0,1067652762,2024-01-02,...,3.0,False,3.0,San Luis Coastal Unified,93401,0.0,11219.0,NaN,CRMLS,CRMLS_CRM
4,CoastalMendocino,CoastalMendocino,Tile,True,NaN,NaN,False,1950000.0,1063453216,2024-01-22,...,NaN,NaN,2.0,NaN,95437,NaN,74487.6,NaN,CRMLS,CRMLS_CRF


In [72]:
df_listings.head()

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1
0,759000.0,1159972295,NaN,NaN,Richard,Sirois,33.738910,-116.198447,42680 Incantata Place,Residential,...,NaN,False,2.0,Desert Sands Unified,92203,NaN,289.0,5227.0,NaN,42680 Incantata Place
1,320000.0,1159871345,2026-05-11,320000.0,Nicholas,Miller,33.842793,-116.481803,28388 Desert Princess Drive,Residential,...,NaN,False,1.0,NaN,92234,Coldwell Banker Realty,855.0,1307.0,NaN,28388 Desert Princess Drive
2,115900.0,1153320466,2026-04-10,160000.0,Heidi,Dunk-Vincent,39.125903,-122.861472,6900 Glenn,Residential,...,3.0,False,1.0,Upper Lake Union,95464,Noble Realty,0.0,3920.0,NaN,6900 Glenn
3,2500000.0,1146529264,2026-04-17,2000000.0,Clifford,Stevens,34.150964,-118.390267,4415 Morella Ave,Residential,...,4.0,False,2.0,Los Angeles Unified,91607,Equity Union,0.0,9906.0,NaN,4415 Morella Ave
4,3200000.0,1118382573,2025-05-21,3200000.0,Trudy,McGrath,33.028788,-117.264434,1200 Cardiff Drive,Residential,...,0.0,False,0.0,San Dieguito Union,92024,NonMember/Member-Other Board,0.0,54885.6,NaN,1200 Cardiff Drive


In [92]:
# Step 1 - Fetch the mortgage rate data from FRED
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates = ["observation_date"])
mortgage.columns = ["date", "rate_30yr_fixed"]

In [93]:
mortgage.head()

,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


In [99]:
# Step 2 - Resample weekly rates to monthly averages
mortgage["year_month"] = mortgage["date"].dt.to_period("M")

mortgage_monthly = mortgage.groupby("year_month")["rate_30yr_fixed"].mean().reset_index()

In [100]:
mortgage_monthly.head()

,year_month,rate_30yr_fixed
0,1971-04,7.3100
1,1971-05,7.4250
2,1971-06,7.5300
3,1971-07,7.6040
4,1971-08,7.6975


In [101]:
# Step 3 - Create a matching year_month key on the MLS datasets

# Sold dataset — key off CloseDate
df_sold["year_month"] = pd.to_datetime(df_sold["CloseDate"]).dt.to_period("M")

# Listings dataset — key off ListingContractDate
df_listings["year_month"] = pd.to_datetime(df_listings["ListingContractDate"]).dt.to_period("M")

In [102]:
df_sold.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,year_month
0,ContraCosta,ContraCosta,"Tile,Wood",NaN,NaN,NaN,False,5000000.0,1095075487,2024-01-18,...,False,3.0,NaN,94595,NaN,30240.0,NaN,CRMLS,CRMLS_CCBE,2024-01
1,NaN,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1079166779,2024-01-30,...,False,2.0,Palm Springs Unified,92262,NaN,13504.0,NaN,CRMLS,CRMLS_MLSL,2024-01
2,Glendale,Southland,NaN,False,NaN,NaN,False,1890500.0,1075037759,2024-01-29,...,False,2.0,Los Angeles Unified,91356,0.0,17873.0,NaN,CRMLS,CRMLS_CRM,2024-01
3,NorthSanLuisObispo,NorthSanLuisObispo,NaN,True,NaN,NaN,False,2100000.0,1067652762,2024-01-02,...,False,3.0,San Luis Coastal Unified,93401,0.0,11219.0,NaN,CRMLS,CRMLS_CRM,2024-01
4,CoastalMendocino,CoastalMendocino,Tile,True,NaN,NaN,False,1950000.0,1063453216,2024-01-22,...,NaN,2.0,NaN,95437,NaN,74487.6,NaN,CRMLS,CRMLS_CRF,2024-01


In [103]:
df_listings.head()

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,year_month
0,759000.0,1159972295,NaN,NaN,Richard,Sirois,33.738910,-116.198447,42680 Incantata Place,Residential,...,False,2.0,Desert Sands Unified,92203,NaN,289.0,5227.0,NaN,42680 Incantata Place,2024-01
1,320000.0,1159871345,2026-05-11,320000.0,Nicholas,Miller,33.842793,-116.481803,28388 Desert Princess Drive,Residential,...,False,1.0,NaN,92234,Coldwell Banker Realty,855.0,1307.0,NaN,28388 Desert Princess Drive,2024-01
2,115900.0,1153320466,2026-04-10,160000.0,Heidi,Dunk-Vincent,39.125903,-122.861472,6900 Glenn,Residential,...,False,1.0,Upper Lake Union,95464,Noble Realty,0.0,3920.0,NaN,6900 Glenn,2024-01
3,2500000.0,1146529264,2026-04-17,2000000.0,Clifford,Stevens,34.150964,-118.390267,4415 Morella Ave,Residential,...,False,2.0,Los Angeles Unified,91607,Equity Union,0.0,9906.0,NaN,4415 Morella Ave,2024-01
4,3200000.0,1118382573,2025-05-21,3200000.0,Trudy,McGrath,33.028788,-117.264434,1200 Cardiff Drive,Residential,...,False,0.0,San Dieguito Union,92024,NonMember/Member-Other Board,0.0,54885.6,NaN,1200 Cardiff Drive,2024-01


In [104]:
# Step 4 - Merge
sold_with_rates = df_sold.merge(mortgage_monthly, on = "year_month", how = "left")
listings_with_rates = df_listings.merge(mortgage_monthly, on = "year_month", how = "left")

In [105]:
sold_with_rates.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,year_month,rate_30yr_fixed
0,ContraCosta,ContraCosta,"Tile,Wood",NaN,NaN,NaN,False,5000000.0,1095075487,2024-01-18,...,3.0,NaN,94595,NaN,30240.0,NaN,CRMLS,CRMLS_CCBE,2024-01,6.6425
1,NaN,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1079166779,2024-01-30,...,2.0,Palm Springs Unified,92262,NaN,13504.0,NaN,CRMLS,CRMLS_MLSL,2024-01,6.6425
2,Glendale,Southland,NaN,False,NaN,NaN,False,1890500.0,1075037759,2024-01-29,...,2.0,Los Angeles Unified,91356,0.0,17873.0,NaN,CRMLS,CRMLS_CRM,2024-01,6.6425
3,NorthSanLuisObispo,NorthSanLuisObispo,NaN,True,NaN,NaN,False,2100000.0,1067652762,2024-01-02,...,3.0,San Luis Coastal Unified,93401,0.0,11219.0,NaN,CRMLS,CRMLS_CRM,2024-01,6.6425
4,CoastalMendocino,CoastalMendocino,Tile,True,NaN,NaN,False,1950000.0,1063453216,2024-01-22,...,2.0,NaN,95437,NaN,74487.6,NaN,CRMLS,CRMLS_CRF,2024-01,6.6425


In [106]:
listings_with_rates.head()

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,...,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,year_month,rate_30yr_fixed
0,759000.0,1159972295,NaN,NaN,Richard,Sirois,33.738910,-116.198447,42680 Incantata Place,Residential,...,2.0,Desert Sands Unified,92203,NaN,289.0,5227.0,NaN,42680 Incantata Place,2024-01,6.6425
1,320000.0,1159871345,2026-05-11,320000.0,Nicholas,Miller,33.842793,-116.481803,28388 Desert Princess Drive,Residential,...,1.0,NaN,92234,Coldwell Banker Realty,855.0,1307.0,NaN,28388 Desert Princess Drive,2024-01,6.6425
2,115900.0,1153320466,2026-04-10,160000.0,Heidi,Dunk-Vincent,39.125903,-122.861472,6900 Glenn,Residential,...,1.0,Upper Lake Union,95464,Noble Realty,0.0,3920.0,NaN,6900 Glenn,2024-01,6.6425
3,2500000.0,1146529264,2026-04-17,2000000.0,Clifford,Stevens,34.150964,-118.390267,4415 Morella Ave,Residential,...,2.0,Los Angeles Unified,91607,Equity Union,0.0,9906.0,NaN,4415 Morella Ave,2024-01,6.6425
4,3200000.0,1118382573,2025-05-21,3200000.0,Trudy,McGrath,33.028788,-117.264434,1200 Cardiff Drive,Residential,...,0.0,San Dieguito Union,92024,NonMember/Member-Other Board,0.0,54885.6,NaN,1200 Cardiff Drive,2024-01,6.6425


In [107]:
# Step 5 - Validate the merge

# Check for any unmatched rows (rate should not be null)
print("Number of unmatched rows (rate_30yr_fixed)")
print(f"- Sold: {sold_with_rates["rate_30yr_fixed"].isnull().sum()}")
print(f"- Listings: {listings_with_rates["rate_30yr_fixed"].isnull().sum()}")

Number of unmatched rows (rate_30yr_fixed)
- Sold: 0
- Listings: 0


In [108]:
sold_with_rates[["CloseDate", "year_month", "ClosePrice", "rate_30yr_fixed"]].head()

,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,2024-01-18,2024-01,5000000.0,6.6425
1,2024-01-30,2024-01,858000.0,6.6425
2,2024-01-29,2024-01,1890500.0,6.6425
3,2024-01-02,2024-01,2100000.0,6.6425
4,2024-01-22,2024-01,1950000.0,6.6425


In [109]:
listings_with_rates[["ListingContractDate", "year_month", "ListPrice.1", "rate_30yr_fixed"]].head()

,ListingContractDate,year_month,ListPrice.1,rate_30yr_fixed
0,2024-01-06,2024-01,659000.0,6.6425
1,2024-01-11,2024-01,320000.0,6.6425
2,2024-01-02,2024-01,149900.0,6.6425
3,2024-01-24,2024-01,2199000.0,6.6425
4,2024-01-28,2024-01,3200000.0,6.6425


In [86]:
sold_with_rates.to_csv("CRMLSSold_3.csv", index = False)
listings_with_rates.to_csv("CRMLSListing_3.csv", index = False)